# Day 6 Exercise 2b - branch, guard, and wire a Strands graph (20 min, auto-graded)

You built the ideas in Demo 2b. Now build the muscle. Six small fills, then run the **scoreboard** at the bottom for a score out of 6.

What you will make (TravelMind, PNR `JX48Q2`):

- a **PNR validator** and a **heuristic extractor** that answer the easy cases for zero model calls
- a **router** that sends each message to write / clarify / escalate
- a **guard** that turns any failure into a safe escalation
- a real **GraphBuilder** wiring of the same routes, which you then **see** and **dry-run**

**Scoring is fully offline. You do not need AWS credentials to pass.** Credentials matter only if you flip `RUN_LIVE = True` at the end to execute the graph for real.

## Before you run

**Colab (3 steps)**
1. `!pip install strands-agents`
2. Nothing else for scoring. AWS creds only if you set `RUN_LIVE = True` later.
3. Run the cells top to bottom, fill the 6 TODOs, then run the scoreboard.

**VS Code (3 steps)**
1. `pip install strands-agents` in your venv.
2. Open this notebook, select your kernel.
3. Fill the TODOs top to bottom, run the scoreboard.

### Why heuristics first (read this once)

A booking system does not need a paid model call to notice the word `refund` or to spot a 6-character code. Cheap rules go first; the model is the fallback for the messy 20 percent.

One trap you will fix in TODO 1: `FLIGHT` is 6 letters. A naive "6 chars" rule proudly reports `FLIGHT` as a PNR. Real record locators mix letters and digits, so the rule must too.

In [ ]:
# ---- provided scaffolding. Pure Python, no AWS. Do not edit. ----
import re
from enum import Enum
from typing import Optional
from pydantic import BaseModel, Field, ValidationError

class Intent(str, Enum):
    BOOKING_STATUS = "booking_status"
    DISRUPTION     = "disruption"
    REBOOKING      = "rebooking"
    REFUND         = "refund"
    BAGGAGE        = "baggage"
    OTHER          = "other"          # non-standard / out of scope -> escalate

class Extraction(BaseModel):
    pnr: Optional[str] = Field(default=None)
    intent: Intent = Field(default=Intent.OTHER)
    confidence: float = Field(default=1.0, ge=0.0, le=1.0)
    ambiguity_reason: Optional[str] = Field(default=None)

class Route(str, Enum):
    WRITE    = "write"
    CLARIFY  = "clarify"
    ESCALATE = "escalate"

PNR_RE = re.compile(r"^[A-Z0-9]{6}$")

INTENT_HINTS = {
    Intent.DISRUPTION:     ("cancel", "cancelled", "delayed", "delay", "disrupt", "stranded"),
    Intent.REBOOKING:      ("rebook", "another flight", "next flight", "change my flight"),
    Intent.REFUND:         ("refund", "money back", "reimburse"),
    Intent.BAGGAGE:        ("baggage", "luggage", "suitcase"),
    Intent.BOOKING_STATUS: ("status", "confirmed", "is my flight"),
}
CANDIDATE = re.compile(r"\b[A-Z0-9]{6}\b")

def find_pnr(text):
    # given: returns the first token that passes YOUR valid_pnr (TODO 1)
    for tok in CANDIDATE.findall(text.upper()):
        if valid_pnr(tok):
            return tok
    return None

# ---- offline helpers for the graph section (used after you finish the TODOs) ----
WIRING = []                 # records (src, dst, route) so we can draw your graph
ENTRY  = {"node": None}

def add_edge_viz(builder, src, dst, route):
    # this is the real Strands call, plus a note for the diagram:
    builder.add_edge(src, dst, condition=route_is(route))   # <-- the line that matters
    WIRING.append((src, dst, route.value))

def set_entry_viz(builder, node):
    builder.set_entry_point(node)                            # real Strands call
    ENTRY["node"] = node

class _MockNode:
    def __init__(self, text): self.result = text
class MockState:
    # mimics a GraphState: .results maps node_id -> object with a .result
    def __init__(self, extract_json): self.results = {"extract": _MockNode(extract_json)}

def _extraction_from_state(state):
    node = state.results.get("extract")
    if not node:
        return None
    try:
        return Extraction.model_validate_json(str(node.result))
    except Exception:
        return None

def dry_run(sample):
    # "runs" the router offline: which single edge fires for this extraction?
    state = MockState(sample.model_dump_json())
    fired = [r for r in (Route.WRITE, Route.CLARIFY, Route.ESCALATE) if route_is(r)(state)]
    branch = fired[0].value if len(fired) == 1 else ("NONE/MULTI " + str([f.value for f in fired]))
    print("  extract --> " + branch +
          "   (intent=" + sample.intent.value + ", pnr=" + str(sample.pnr) + ", conf=" + str(sample.confidence) + ")")

def show_graph():
    print("entry:", ENTRY["node"])
    for s, d, r in WIRING:
        print("  " + s + " --[" + r + "]--> " + d)
    print("\nmermaid (paste into any Mermaid viewer):\n")
    lines = ["flowchart TD",
             '    extract["extract: structured output"]',
             '    write["write: happy path"]',
             '    clarify["clarify: ask 1 question"]',
             '    escalate["escalate: human"]']
    for s, d, r in WIRING:
        lines.append("    " + s + " -->|" + r + "| " + d)
    print("\n".join(lines))

SAMPLES = [
    Extraction(pnr="JX48Q2", intent=Intent.BOOKING_STATUS, confidence=0.95),   # should -> write
    Extraction(pnr=None,     intent=Intent.DISRUPTION,     confidence=0.90),   # should -> clarify
    Extraction(pnr="JX48Q2", intent=Intent.OTHER,          confidence=0.90),   # should -> escalate
]

print("scaffolding ready")

### TODO 1 - `valid_pnr`
A real PNR is exactly 6 characters, matches `PNR_RE`, and has at least one letter and one digit.

In [ ]:
def valid_pnr(pnr):
    if not pnr:
        return False
    p = pnr.strip().upper()
    # TODO 1: extend the line below so it ALSO requires >=1 digit and >=1 letter.
    #   hint: any(c.isdigit() for c in p)  and  any(c.isalpha() for c in p)
    return bool(PNR_RE.match(p))   # <-- extend this

# quick check
print(valid_pnr("JX48Q2"), valid_pnr("FLIGHT"), valid_pnr("123456"))   # want: True False False

### TODO 2 - `cheap_extract` (the guard on the heuristic)
Trust the rule only when it is unambiguous: exactly one intent hit AND a valid PNR. When unsure, return `None` so the model can take over.

In [ ]:
def cheap_extract(msg):
    pnr  = find_pnr(msg)                                          # given
    hits = [i for i, words in INTENT_HINTS.items()
            if any(w in msg.lower() for w in words)]              # given: list of matched intents
    # TODO 2: replace the condition so we return an Extraction ONLY when
    #   there is exactly one intent hit AND pnr is valid. Otherwise fall through to None.
    if False:   # <-- replace this condition
        return Extraction(pnr=pnr, intent=hits[0], confidence=0.95)
    return None

print(cheap_extract("Is my flight confirmed? PNR JX48Q2"))                 # want: an Extraction
print(cheap_extract("My flight got cancelled, what now?"))                 # want: None (no valid pnr)
print(cheap_extract("I want a refund and it was cancelled, PNR JX48Q2"))   # want: None (2 intents)

### TODO 3 - `decide_route` (the branch)
This is the heart of the workflow. Apply the rules in order.

```
                 intent == OTHER            -> ESCALATE   (non-standard, red flag)
   Extraction -> pnr not valid              -> CLARIFY    (recover the PNR)
                 ambiguity or low confidence-> CLARIFY
                 otherwise                  -> WRITE      (happy path)
```

In [ ]:
def decide_route(ex: Extraction) -> Route:
    # TODO 3: return the right Route, checking the rules IN ORDER:
    #   1) ex.intent == Intent.OTHER                         -> Route.ESCALATE
    #   2) not valid_pnr(ex.pnr)                              -> Route.CLARIFY
    #   3) ex.ambiguity_reason or ex.confidence < 0.6        -> Route.CLARIFY
    #   4) everything checks out                             -> Route.WRITE
    return Route.WRITE   # <-- replace with the four rules above

# quick check
mk = lambda **k: Extraction(**{"pnr": "JX48Q2", "intent": Intent.BOOKING_STATUS, "confidence": 0.95, **k})
print(decide_route(mk(intent=Intent.OTHER)), decide_route(mk(pnr=None)), decide_route(mk()))
# want: Route.ESCALATE Route.CLARIFY Route.WRITE

### TODO 4 - `guard` (fail safe)
Run a stage inside a try/except so one bad call never crashes the request. On success record `("ok")` and return the value; on failure record the error type and return nothing.

In [ ]:
def guard(label, fn, trace):
    # TODO 4: fill the two branches.
    #   success -> trace.append((label, "ok"));  return (True, value)
    #   failure -> trace.append((label, "error:" + type(err).__name__));  return (False, None)
    # in production the errors you expect here are EventLoopException,
    # ContextWindowOverflowException, MaxTokensReachedException, and pydantic ValidationError.
    try:
        pass   # <-- call fn(), record ok, return (True, value)
    except Exception as err:
        pass   # <-- record the error, return (False, None)

# quick check
tr = []
print(guard("a", lambda: 42, tr), tr)              # want: (True, 42) [('a', 'ok')]

### TODO 5 - `route_is` (the graph condition)
A Strands conditional edge is just a function that reads the graph state and returns True or False. Reuse your `decide_route`. If the extractor produced junk (`ex is None`), only the escalate edge should fire.

In [ ]:
def route_is(target: Route):
    def _cond(state):
        ex = _extraction_from_state(state)   # given: parses the extract node's output, or None
        # TODO 5: fix the two returns below.
        if ex is None:
            return False   # <-- on junk, which target should fire?  (compare target with Route.ESCALATE)
        return False       # <-- otherwise fire when decide_route(ex) equals target
    return _cond

# quick check
sW = MockState(Extraction(pnr="JX48Q2", intent=Intent.BOOKING_STATUS, confidence=0.95).model_dump_json())
print(route_is(Route.WRITE)(sW), route_is(Route.CLARIFY)(sW))   # want: True False

### TODO 6 - wire the graph
Add the three conditional edges from `extract`, then set the entry point. Building the graph needs Strands installed but makes no AWS call.

In [ ]:
from strands import Agent
from strands.models import BedrockModel
from strands.multiagent import GraphBuilder

REGION = "us-west-2"
MODEL  = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"
bedrock = BedrockModel(model_id=MODEL, region_name=REGION, temperature=0.2)   # constructs offline

# four node agents (given). They are only executed if you set RUN_LIVE = True later.
n_extract  = Agent(model=bedrock, callback_handler=None, system_prompt="Return ONLY JSON for the extraction.")
n_write    = Agent(model=bedrock, system_prompt="Write a warm 2-sentence reply from the JSON details.")
n_clarify  = Agent(model=bedrock, system_prompt="Ask ONE specific question to recover the missing detail.")
n_escalate = Agent(model=bedrock, system_prompt="Say the issue is going to a human specialist, in 2 sentences.")

builder = GraphBuilder()
builder.add_node(n_extract,  "extract")     # given
builder.add_node(n_write,    "write")
builder.add_node(n_clarify,  "clarify")
builder.add_node(n_escalate, "escalate")

# TODO 6: wire three conditional edges from "extract", then set the entry point.
#   use the helpers so your graph also becomes drawable:
# add_edge_viz(builder, "extract", "write",    Route.____)
# add_edge_viz(builder, "extract", "clarify",  Route.____)
# add_edge_viz(builder, "extract", "escalate", Route.____)
# set_entry_viz(builder, "____")


graph = builder.build()   # given
print("graph built:", graph is not None, "| edges wired:", len(WIRING), "| entry:", ENTRY["node"])

### See your graph, then dry-run the routing (offline)
`show_graph` prints your topology and a Mermaid diagram. `dry_run` feeds sample extractions through your `route_is` edges and prints which branch fires, with no Bedrock call.

In [ ]:
show_graph()
print("\n--- dry run ---")
for s in SAMPLES:
    dry_run(s)
# want: write, clarify, escalate (one per line)

### Optional stretch: run it for real
Needs AWS creds and Sonnet 4.5 access in `us-west-2`. Leave `RUN_LIVE = False` to skip.

In [ ]:
RUN_LIVE = False
if RUN_LIVE:
    graph("Is my flight confirmed? PNR JX48Q2")   # nodes stream their reply; one branch fires
else:
    print("RUN_LIVE is False - skipping the real Bedrock call. The dry run above already showed routing.")

### Scoreboard
Run this last. It asserts each TODO and prints your score out of 6.

In [ ]:
# ---- assert-based scoreboard. Run after the TODOs. ----
score = 0; total = 0
def check(label, test):
    global score, total
    total += 1
    try:
        test(); score += 1; print("  PASS  " + label)
    except Exception as e:
        print("  FAIL  " + label + "  ->  " + (str(e) or type(e).__name__))

def _raise():
    raise RuntimeError("boom")

def t1():
    assert valid_pnr("JX48Q2") and valid_pnr("ABC123"), "valid mixed PNRs must pass"
    assert not valid_pnr("FLIGHT"), "6 letters, no digit -> reject"
    assert not valid_pnr("123456"), "6 digits, no letter -> reject"
    assert not valid_pnr("JX4Q") and not valid_pnr(None), "wrong length / None -> reject"

def t2():
    e = cheap_extract("Is my flight confirmed? PNR JX48Q2")
    assert e is not None and e.pnr == "JX48Q2" and e.intent == Intent.BOOKING_STATUS, "clean case should extract"
    assert cheap_extract("My flight got cancelled, what now?") is None, "no valid pnr -> None"
    assert cheap_extract("I want a refund and it was cancelled, PNR JX48Q2") is None, "two intents -> None"

def t3():
    mk = lambda **k: Extraction(**{"pnr": "JX48Q2", "intent": Intent.BOOKING_STATUS, "confidence": 0.95, **k})
    assert decide_route(mk(intent=Intent.OTHER)) == Route.ESCALATE
    assert decide_route(mk()) == Route.WRITE
    assert decide_route(mk(pnr=None)) == Route.CLARIFY
    assert decide_route(mk(ambiguity_reason="two bookings")) == Route.CLARIFY
    assert decide_route(mk(confidence=0.3)) == Route.CLARIFY

def t4():
    tr = []
    ok, v = guard("a", lambda: 42, tr)
    assert ok and v == 42 and tr[-1] == ("a", "ok"), "success path must record ok and return the value"
    ok, v = guard("b", _raise, tr)
    assert (not ok) and v is None and tr[-1][0] == "b" and "error" in tr[-1][1], "failure path must be caught"

def t5():
    sW = MockState(Extraction(pnr="JX48Q2", intent=Intent.BOOKING_STATUS, confidence=0.95).model_dump_json())
    assert route_is(Route.WRITE)(sW) and not route_is(Route.CLARIFY)(sW) and not route_is(Route.ESCALATE)(sW)
    sBad = MockState("not json")
    assert route_is(Route.ESCALATE)(sBad) and not route_is(Route.WRITE)(sBad), "junk state -> only escalate fires"

def t6():
    routes = {(s, d): r for s, d, r in WIRING}
    assert routes.get(("extract", "write")) == "write", "extract -> write on WRITE"
    assert routes.get(("extract", "clarify")) == "clarify", "extract -> clarify on CLARIFY"
    assert routes.get(("extract", "escalate")) == "escalate", "extract -> escalate on ESCALATE"
    assert ENTRY["node"] == "extract", "entry point must be extract"
    assert graph is not None, "builder.build() must succeed"

for lbl, fn in [("TODO 1  valid_pnr rejects word-shaped PNRs", t1),
                ("TODO 2  cheap_extract only trusts the sure cases", t2),
                ("TODO 3  decide_route branches correctly", t3),
                ("TODO 4  guard catches failures", t4),
                ("TODO 5  route_is reads state and reuses decide_route", t5),
                ("TODO 6  graph wired: 3 edges + entry point", t6)]:
    check(lbl, fn)

print("\nSCORE: " + str(score) + " / " + str(total))
print("All checks passed - nice work." if score == total else "Fix the FAILs above, then re-run this cell.")